# 01 — Data Audit: Section 1, Structural Checks

This notebook works through the **Structural** checks from §4 of the blueprint (`notebooks/subsea_sentinel_blueprint.md`). Before we build any features or models, we need to prove the raw file is what we think it is: the right shape, no duplicate observations, the right year range, and — because this is a *panel* (repeated observations of the same cables over time) — that it behaves like a real-world panel where entities can enter and exit, with the exception of six \"gappy\" cables we need to find and set aside later.

**Scope of this notebook (for now):** only the structural checks below. Target integrity, derived-column consistency, variable-meaning checks, and generator-artifact checks come later, as separate sections.

What we'll check, in order:
1. Row count is 4,821; 500 unique `cable_id` values
2. `cable_id` × `year` combinations are unique (no duplicate observations)
3. Years span 2015–2026
4. The panel is unbalanced (some cables enter after 2015, some exit before 2026)
5. Find the 6 cables with year gaps in their history"


## Setup

We use `pandas` (a library for working with tables of data, similar to a spreadsheet but scriptable) to load the CSV into a `DataFrame` — pandas' name for a table. Each row of `undersea_cables_master.csv` is one cable observed in one year."

In [ ]:
import pandas as pd

df = pd.read_csv('../data/undersea_cables_master.csv')
df = df.sort_values(['cable_id', 'year']).reset_index(drop=True)

df.head()

## Check 1 — Row count and unique cable count

**What we're testing:** the file should have exactly 4,821 rows (one row per cable per year it was observed), and exactly 500 distinct cables should be responsible for those rows.

**Why it matters:** everything downstream — feature engineering, train/test splits, model evaluation — assumes we know how many cables and how many cable-year observations exist. If the row count is off, either the file got truncated/duplicated on load, or our understanding of \"one row = one cable-year\" is wrong. Catching that now, with a loud `assert`, is much cheaper than discovering it after building features on top of bad data."

In [ ]:
n_rows = len(df)
n_unique_cables = df['cable_id'].nunique()

print(f'Row count: {n_rows}')
print(f'Unique cable_id count: {n_unique_cables}')

assert n_rows == 4_821, f'Expected 4821 rows, got {n_rows}'
assert n_unique_cables == 500, f'Expected 500 unique cables, got {n_unique_cables}'
print('PASS: row count and unique cable count match expectations.')

## Check 2 — `cable_id` × `year` is unique

**What we're testing:** for every (cable, year) pair, there should be at most one row. In other words, we should never see `CAB0001` appear twice for `2018`.

**Why it matters:** this is the *primary key* of the panel — the combination of columns that uniquely identifies a row. Later on, we'll do things like `groupby('cable_id').shift()` to get \"last year's fault status\" for each cable. If a (cable, year) pair were duplicated, that operation would silently produce two rows of history for one real year, corrupting every history-based feature downstream without raising an error."

In [ ]:
n_duplicate_keys = df.duplicated(subset=['cable_id', 'year']).sum()

print(f'Number of duplicated (cable_id, year) rows: {n_duplicate_keys}')

assert n_duplicate_keys == 0, 'Found duplicate (cable_id, year) rows — panel key is not unique!'
print('PASS: every (cable_id, year) pair appears exactly once.')

## Check 3 — Years span 2015–2026

**What we're testing:** the `year` column's minimum is 2015 and its maximum is 2026, with no years outside that range.

**Why it matters:** the entire modeling plan (§7 of the blueprint) is built around specific year ranges — dev folds from 2019–2024, a frozen holdout at 2025, and forward scoring for 2026. If the file actually contained, say, an extra row from 2027 or was missing 2026 entirely, every one of those downstream decisions would need to change. We confirm the boundary now so we can treat it as fixed later."

In [ ]:
min_year = df['year'].min()
max_year = df['year'].max()
years_present = sorted(df['year'].unique().tolist())

print(f'Min year: {min_year}')
print(f'Max year: {max_year}')
print(f'All years present: {years_present}')

assert min_year == 2015, f'Expected min year 2015, got {min_year}'
assert max_year == 2026, f'Expected max year 2026, got {max_year}'
print('PASS: years span 2015-2026.')

## Check 4 — The panel is unbalanced

**What we're testing:** a \"balanced panel\" would mean every one of the 500 cables has exactly one row for every year from 2015 to 2026 (12 rows each, 500 × 12 = 6,000 rows total). We only have 4,821 rows, so it can't be balanced — but we want to confirm *why*: some cables' first observed year is later than 2015 (they entered service after the panel started), and some cables' last observed year is earlier than 2026 (they exited — e.g. decommissioned, or simply not observed further). The blueprint's expected numbers are **188 cables entering after 2015** and **18 cables exiting before 2026**.

**Why it matters:** an unbalanced panel is completely normal for real infrastructure (cables get built and retired over time), but it has a direct consequence for modeling: we can't assume every cable has 12 years of history. Features like \"3-year rolling fault count\" need to handle cables with only 2 years of data. Confirming *how* unbalanced the panel is (which cables, how many) tells us how much of that edge-case handling we'll need."

In [ ]:
balanced_row_count = n_unique_cables * len(years_present)
print(f'A balanced panel would have {n_unique_cables} cables x {len(years_present)} years = {balanced_row_count} rows')
print(f'Actual row count: {n_rows} -> panel is unbalanced by {balanced_row_count - n_rows} rows\
')

# For each cable, find the first (min) and last (max) year it appears in the data.
# groupby('cable_id') splits the table into one mini-table per cable; ['year'] picks
# out just the year column from each; .agg(['min', 'max']) computes both the minimum
# and maximum year for each cable in one pass, returning a table indexed by cable_id
# with 'min' and 'max' columns.
cable_span = df.groupby('cable_id')['year'].agg(['min', 'max'])

entered_after_2015 = (cable_span['min'] > 2015).sum()
exited_before_2026 = (cable_span['max'] < 2026).sum()

print(f'Cables entering after 2015: {entered_after_2015}')
print(f'Cables exiting before 2026: {exited_before_2026}')

assert entered_after_2015 == 188, f'Expected 188 late entrants, got {entered_after_2015}'
assert exited_before_2026 == 18, f'Expected 18 early exits, got {exited_before_2026}'
print('PASS: panel is unbalanced as expected (188 late entrants, 18 early exits).')

## Check 5 — Find the 6 cables with year gaps

**What we're testing:** entering-late and exiting-early (Check 4) both produce a cable with a *contiguous* run of years — e.g. a cable observed 2018–2023 has 6 rows, and `max_year - min_year + 1 = 2023 - 2018 + 1 = 6`, which matches its row count. A \"gap\" cable is different: it's missing one or more years in the *middle* of its span. For example, a cable observed in every year from 2015–2024 and then again in 2026 (skipping 2025) has 11 rows, but `max_year - min_year + 1 = 2026 - 2015 + 1 = 12`. The row count (11) no longer matches the span (12) — that mismatch is exactly what exposes a gap.

**Why it matters:** the blueprint flags this explicitly (§0, correction #5 and §6, Step 4): a gap silently breaks any \"rolling window\" or \"previous year\" feature we build later, because pandas has no idea a year is missing — `groupby('cable_id').shift(1)` will happily hand you 2024's data and call it \"last year's value\" even when the actual previous row is from 2023. Finding these 6 cables now means we can make every history feature \"year-aware\" later, rather than discovering silently wrong numbers after modeling."

In [ ]:
# .size() counts how many rows fall into each group -- here, how many rows each
# cable_id has. This is our actual row count per cable.
cable_row_counts = df.groupby('cable_id').size()

# Reuse cable_span (min/max year per cable) from Check 4 and attach the row count
# and the \"expected\" row count if there were no gaps (span from min to max year,
# inclusive, hence the +1: e.g. 2015-2017 is 3 years, and 2017-2015+1 = 3).
cable_span = cable_span.copy()
cable_span['row_count'] = cable_row_counts
cable_span['expected_span'] = cable_span['max'] - cable_span['min'] + 1
cable_span['has_gap'] = cable_span['expected_span'] != cable_span['row_count']

gap_cable_ids = cable_span[cable_span['has_gap']].index.tolist()

print(f'Cables with year gaps: {len(gap_cable_ids)}\
')

assert len(gap_cable_ids) == 6, f'Expected 6 gap cables, found {len(gap_cable_ids)}'

for cable_id in gap_cable_ids:
    # Pull every year this specific cable was observed in, sorted low to high,
    # so we can see exactly where the missing year falls.
    cable_years = sorted(df.loc[df['cable_id'] == cable_id, 'year'])
    print(f'{cable_id}: {cable_years}')

print('\
PASS: found exactly 6 cables whose year range does not match their row count.')

## Section 1 summary

All five structural checks pass: 4,821 rows across 500 uniquely-identified cables, no duplicate (cable_id, year) rows, years spanning 2015–2026, an unbalanced panel (188 late entrants, 18 early exits), and 6 cables with a mid-history gap. The gap cables are not fixed here — per the blueprint (§6, Step 5), they get dropped later once we build the target column, since a gap breaks the `fault_next_year` label for the row right before the missing year.

## Section 2 — Target Integrity

This section works through the second checklist in §4 of the blueprint: making sure `fault_next_year` — the column we're eventually trying to predict — means what its name says it means. The gap cables from Section 1 turn out to matter here too: for exactly those 6 cables, `fault_next_year` silently describes a fault two years out instead of one.

What we'll check, in order:
6. `fault_next_year` is NaN for exactly 500 rows (each cable's last observation)
7. For all other rows, `fault_next_year` equals the next row's `fault_this_year`
8. List the 6 rows where that "next row" is not actually `year + 1`
9. Overall positive rate and prevalence by year

**Scope of this section:** find and document these problems only. Per the blueprint (§6, Step 5), rows are actually dropped later, in `src/preprocessing.py` — not here. Section 3 (derived-column consistency) is not started in this notebook.

## Check 6 — `fault_next_year` is NaN for exactly 500 rows

**What we're testing:** `fault_next_year` should be missing (`NaN`) for exactly 500 rows — and those 500 rows should be exactly each cable's *last* observed year, no more and no less.

**Why it matters:** `fault_next_year` means "did this cable fault the year after this row." For a cable's most recent observation, there is no recorded "year after" anywhere in the dataset, so the value has to be missing — the data isn't broken, the question is just unanswerable for that row. If the NaNs showed up anywhere else, or didn't line up one-to-one with the 500 cables' last rows, that would mean the target column was built incorrectly, and every model trained on it would be learning from a wrong label.

In [ ]:
nan_count = df['fault_next_year'].isna().sum()
print(f"Rows where fault_next_year is NaN: {nan_count}")
assert nan_count == 500, f"Expected 500 NaN rows, got {nan_count}"

# For each cable, find the row index of its LAST (max-year) observation.
# groupby('cable_id')['year'].idxmax() returns, per cable, the original
# DataFrame row label where that cable's year is largest -- not the year
# itself, but a pointer to the row.
last_row_idx = df.groupby('cable_id')['year'].idxmax()
is_last_row = df.index.isin(last_row_idx)

# Checking both directions -- every NaN row is a last row, AND every last
# row is NaN -- confirms an exact match, not just an overlap.
nan_rows_are_last_rows = (df['fault_next_year'].isna() == is_last_row).all()
print(f"Every NaN row is a cable's last row, and vice versa: {nan_rows_are_last_rows}")

assert nan_rows_are_last_rows, "Mismatch between NaN rows and cable-last rows!"
print("PASS: fault_next_year is NaN for exactly the 500 cable-last rows.")

## Check 7 — `fault_next_year` equals the next row's `fault_this_year`

**What we're testing:** for every row where `fault_next_year` isn't missing, its value should equal `fault_this_year` from *the next row for that same cable* — i.e., the target was built by looking one row ahead, not some other way.

**Why it matters:** this is the mechanical definition check — seeing with our own eyes that the label really is "next row's fault status," using the same `groupby` + `shift` pattern we'll rely on constantly once we build features (blueprint §6, Step 4). We use `shift(-1)` here, not `shift(1)`: `shift(1)` pulls a value from the *previous* row (that's what "last year's value" features will use later); `shift(-1)` pulls from the *next* row, which is what we need to reconstruct `fault_next_year` and compare it.

**A subtlety worth flagging before you run this:** this check compares row *values*, not row *years*. It will pass for every single row — including the 6 gap-affected rows from Section 1 — because the column was built by literally copying the next row's `fault_this_year`, regardless of whether that next row was really next year. That's exactly the bug: a value-only check can't see it. Check 8 adds the missing piece by comparing years, not just values.

In [ ]:
# shift(-1) within each cable's group pulls the value from the NEXT row for
# that cable. Because we group by cable_id first, this never reaches across
# into a different cable's data -- pandas keeps the shift confined to each
# group, and puts NaN at the last row of every group (nothing to shift in).
next_row_fault = df.groupby('cable_id')['fault_this_year'].shift(-1)

non_nan = df[df['fault_next_year'].notna()].copy()
non_nan['next_row_fault'] = next_row_fault[non_nan.index]

value_mismatches = non_nan[non_nan['fault_next_year'] != non_nan['next_row_fault']]
print(f"Non-NaN rows checked: {len(non_nan)}")
print(f"Rows where fault_next_year != next row's fault_this_year: {len(value_mismatches)}")

assert len(value_mismatches) == 0, "fault_next_year does not match the next row's fault_this_year!"
print(f"PASS: fault_next_year equals the next row's fault_this_year for all {len(non_nan)} non-NaN rows.")
print("(This includes the 6 gap cables -- see Check 8 for why that is itself the problem.)")

## Check 8 — List the 6 rows where the "next row" isn't actually next year

**What we're testing:** for each non-NaN row, does the row `shift(-1)` pulled `fault_this_year` from actually come from `year + 1`? For 6 rows — the row right before each gap cable's missing year — the answer is no: the "next row" in the sorted table skips straight past the gap to a later year.

**Why it matters:** Check 7 proved the column is built *positionally* ("next row in the table"). This check proves that, for 6 specific rows, "next row" and "next year" are not the same thing — so `fault_next_year` on those 6 rows is quietly describing a fault two years later, not one. The blueprint (§0, correction #5) calls this out explicitly and requires these 6 rows to be dropped before modeling. This notebook only finds and lists them — dropping them happens later, in `src/preprocessing.py`.

In [ ]:
# Same shift(-1) idea as Check 7, but on the year column instead of the
# fault column -- this tells us what year the next row actually belongs to.
next_row_year = df.groupby('cable_id')['year'].shift(-1)
non_nan['next_row_year'] = next_row_year[non_nan.index]

mislabeled = non_nan[non_nan['next_row_year'] != non_nan['year'] + 1]

print(f"Rows where the next row isn't year + 1: {len(mislabeled)}")
print()
assert len(mislabeled) == 6, f"Expected 6 mislabeled rows, found {len(mislabeled)}"

for _, row in mislabeled.iterrows():
    labeled_year = int(row['year'])
    actual_year = int(row['next_row_year'])
    claimed_year = labeled_year + 1
    print(
        f"{row['cable_id']}: the fault_next_year recorded on the {labeled_year} row "
        f"claims to describe {claimed_year} (since {claimed_year} is missing for this "
        f"cable, per Section 1's gap finding), but the next real observation is "
        f"{actual_year} -- so it actually describes a fault in {actual_year}."
    )

print()
print('PASS: found exactly 6 mislabeled rows, matching the 6 gap cables from Section 1.')

## Check 9 — Overall positive rate and prevalence by year

**What we're testing:** what fraction of non-NaN `fault_next_year` rows are `1` (a fault happened the following year) — overall, and broken out by year. The blueprint expects an overall rate near **0.190**, with the by-year figures ranging from about **0.166 to 0.219**.

**Why it matters:** this is a *rare-event* problem — only about 1 in 5 observations is a positive case — which is why the blueprint's metrics section (§10) uses PR-AUC instead of plain accuracy (a model that always predicts "no fault" would already be right about 81% of the time). The by-year table matters even more: prevalence isn't constant across years, so a raw PR-AUC from an easy year (higher prevalence) isn't comparable to one from a hard year (lower prevalence). That's exactly what "PR lift" (§10) divides out later — which is why we keep the full, unrounded `groupby` output below instead of rounding early and losing precision we'll need again.

In [ ]:
overall_rate = df['fault_next_year'].mean()
print(f"Overall positive rate: {overall_rate}")
assert abs(overall_rate - 0.190) < 0.01, f"Expected ~0.190, got {overall_rate}"

# groupby('year') splits rows by the CURRENT observation year; the mean of
# fault_next_year within each group is the fraction of THAT year's cables
# that went on to fault the FOLLOWING year -- i.e., target prevalence for
# that fold. 2026 rows are all NaN (there's no "next year" to report), so
# its mean comes back as NaN and its count as 0 -- expected, not a bug.
prevalence_by_year = df.groupby('year')['fault_next_year'].agg(['mean', 'sum', 'count'])

by_year_rates = prevalence_by_year['mean'].dropna()
print(f"Prevalence by year ranges from {by_year_rates.min()} to {by_year_rates.max()}")
assert by_year_rates.min() > 0.16 and by_year_rates.max() < 0.22, "By-year prevalence outside expected range"

prevalence_by_year

## Section 2 summary

All four target-integrity checks pass: `fault_next_year` is NaN for exactly the 500 cable-last rows; for every other row it equals the next row's `fault_this_year` by construction (Check 7); and for exactly 6 rows — one per gap cable from Section 1 — that "next row" turns out to be two years out instead of one, so the label silently describes the wrong year (Check 8). The overall positive rate is ~0.190, with by-year prevalence ranging from about 0.166 to 0.219 (table above), which we'll reuse for PR lift later.

No rows were dropped in this notebook. Per the blueprint (§6, Step 5), the 500 NaN rows and the 6 mislabeled rows get removed in `src/preprocessing.py`, not here.

**Section 3 (derived-column consistency) is not started in this notebook.**

## Section 3 — Derived-Column Consistency

This section works through the third checklist in §4 of the blueprint. A "derived column" is one that could, in principle, be computed from other columns already in the table (e.g. `age_years` should just be `year - rfs_year`). Just because a column *could* be computed that way doesn't mean it *was* — data generators (and real-world pipelines) sometimes drift, so we check each relationship directly instead of assuming it holds.

What we'll check, in order:
10. `age_years == year - rfs_year` for every row
11. `lit_capacity_tbps <= design_capacity_tbps` for every row (you can't light more fiber than exists)
12. Whether `utilization_pct` is really the same thing as `lit_capacity_tbps / design_capacity_tbps * 100` (spoiler: it isn't — we need the actual correlation and gap numbers to decide which one to keep as a model feature later)
13. `design_life_years` is constant across the whole dataset
14. `fault_cause == 'none'` whenever `fault_this_year == 0`, with zero exceptions

**Scope of this section:** find and document these relationships only. Section 4 (variable meaning) is not started in this notebook.

## Check 10 — `age_years == year - rfs_year`

**What we're testing:** `rfs_year` is the year a cable went "ready for service" (i.e. went live). `age_years` should just be how many years have passed since then: `year - rfs_year`. We check that every row's stored `age_years` matches this formula exactly.

**Why it matters:** if this holds with zero mismatches, `age_years` carries no information beyond what `year` and `rfs_year` already give us — it's a convenience column, not new data. That matters for modeling later: the blueprint's feature dictionary (§5) drops `rfs_year` from the linear model precisely because it's collinear with `age_years` and `year` together — this check is what justifies that decision.

In [ ]:
expected_age = df['year'] - df['rfs_year']
age_mismatches = df[df['age_years'] != expected_age]

print(f"Rows where age_years != year - rfs_year: {len(age_mismatches)}")

assert len(age_mismatches) == 0, f"Found {len(age_mismatches)} age_years mismatches!"
print("PASS: age_years == year - rfs_year holds for every row.")

## Check 11 — `lit_capacity_tbps <= design_capacity_tbps`

**What we're testing:** `design_capacity_tbps` is the maximum capacity the cable was engineered for; `lit_capacity_tbps` is how much of that capacity is actually turned on and in use. Physically, you cannot light up more fiber capacity than exists — so `lit_capacity_tbps` should never exceed `design_capacity_tbps`, for any row.

**Why it matters:** this is a sanity check on the data-generating process, not just a nice-to-have. If it were violated, it would mean either the two columns aren't measuring what their names claim, or there's a generator bug — and any feature built from their ratio (like `lit / design`, which the blueprint calls `cap_ratio` in §5) would produce nonsensical values above 100%.

In [ ]:
cap_violations = df[df['lit_capacity_tbps'] > df['design_capacity_tbps']]

print(f"Rows where lit_capacity_tbps > design_capacity_tbps: {len(cap_violations)}")

assert len(cap_violations) == 0, f"Found {len(cap_violations)} capacity violations!"
print("PASS: lit_capacity_tbps never exceeds design_capacity_tbps.")

## Check 12 — `utilization_pct` vs. the implied capacity ratio

**What we're testing:** you might expect `utilization_pct` to just be `lit_capacity_tbps / design_capacity_tbps * 100` restated as a percentage. We test that directly by computing the implied ratio ourselves and comparing it to the `utilization_pct` column that's already in the file — using **correlation** (does one go up when the other goes up?) and the **gap** between them (`utilization_pct` minus the implied ratio: its mean, minimum, and maximum).

**Why it matters:** correlation of `1.0` would mean they're the same quantity wearing two names — in that case we'd only need one. A correlation well below `1.0`, combined with a gap that swings both positive and negative, means `utilization_pct` was generated as a *separate, noisier* quantity, not derived from the capacity columns. The blueprint (§5) says to keep only one of `utilization_pct` or `cap_ratio` (`lit / design`) as a feature, and to justify the choice — the actual numbers below are what that justification is built on, so we print them at full precision instead of rounding them away.

In [ ]:
# The ratio utilization_pct would equal IF it were just derived from the two
# capacity columns. This is not a real column in the CSV -- we build it here
# purely to compare against the real utilization_pct column.
implied_utilization_pct = df['lit_capacity_tbps'] / df['design_capacity_tbps'] * 100

# .corr() computes the Pearson correlation coefficient: a number from -1 to 1
# measuring how closely two columns move together in a straight-line sense.
# 1.0 would mean they're perfectly interchangeable; 0.0 would mean no linear
# relationship at all.
correlation = df['utilization_pct'].corr(implied_utilization_pct)

# The gap is just one column minus the other, row by row -- a Series with one
# value per row, which we then summarize with .mean() / .min() / .max().
gap = df['utilization_pct'] - implied_utilization_pct

print(f"Correlation (utilization_pct vs. implied lit/design ratio): {correlation}")
print(f"Gap mean (utilization_pct - implied):  {gap.mean()}")
print(f"Gap min:                               {gap.min()}")
print(f"Gap max:                               {gap.max()}")

print()
print("CONCLUSION: correlation is moderate (nowhere near 1.0), and the gap swings")
print("both strongly negative and strongly positive across rows -- utilization_pct")
print("and the implied lit/design capacity ratio are NOT the same quantity.")

## Check 13 — `design_life_years` has exactly one unique value

**What we're testing:** the data dictionary describes `design_life_years` as "engineered design life (~25y)." We check whether it's actually a single fixed number across all 4,821 rows, or whether it varies by cable.

**Why it matters:** a column with only one distinct value carries zero information for telling rows apart — it can't help a model distinguish a high-risk cable from a low-risk one, no matter how it's used. The blueprint's feature dictionary (§5) marks `design_life_years` for removal on exactly this basis ("Constant = 25"); this check is what confirms that call is justified rather than assumed.

In [ ]:
unique_design_life_values = df['design_life_years'].unique()
n_unique_design_life = df['design_life_years'].nunique()

print(f"Unique design_life_years values: {unique_design_life_values}")
print(f"Number of unique values: {n_unique_design_life}")

assert n_unique_design_life == 1, f"Expected 1 unique value, found {n_unique_design_life}"
assert unique_design_life_values[0] == 25, f"Expected the value 25, got {unique_design_life_values[0]}"
print("PASS: design_life_years is constant at 25 across every row.")

## Check 14 — `fault_cause == 'none'` whenever `fault_this_year == 0`

**What we're testing:** `fault_cause` records *why* a cable faulted (`fishing_anchor`, `natural_hazard`, `equipment`, `suspected_external`) or `'none'` if it didn't fault that year. We check that every single row with `fault_this_year == 0` (no fault) has `fault_cause == 'none'` — with zero exceptions.

**Why it matters:** if this holds perfectly, `fault_cause` doesn't just correlate with `fault_this_year` — it's a deterministic function of it. A column that perfectly determines the label is called a **leakage proxy**: a model given `fault_cause` wouldn't need to learn anything about real risk factors, because it could just check "is this `'none'` or not?" and get the answer for free. That's not a legitimate predictive signal — it's the answer sheet in disguise. This is exactly why the blueprint's feature dictionary (§5) marks `fault_cause` **Remove**, and why Step 3 of the preprocessing sequence (§6) drops it before any model ever sees the data.

In [ ]:
no_fault_rows = df[df['fault_this_year'] == 0]
exceptions = no_fault_rows[no_fault_rows['fault_cause'] != 'none']

print(f"Rows with fault_this_year == 0: {len(no_fault_rows)}")
print(f"Of those, rows where fault_cause != 'none': {len(exceptions)}")

assert len(exceptions) == 0, f"Found {len(exceptions)} exceptions to fault_cause determinism!"
print("PASS: fault_cause == 'none' for every row where fault_this_year == 0, with zero exceptions.")
print()
print(f"All fault_cause values seen when fault_this_year == 1: {sorted(df.loc[df['fault_this_year'] == 1, 'fault_cause'].unique())}")

## Section 3 summary

All five derived-column checks pass or resolve as expected: `age_years` exactly equals `year - rfs_year` (0 mismatches); `lit_capacity_tbps` never exceeds `design_capacity_tbps` (0 violations); `utilization_pct` and the implied `lit/design` ratio are only moderately correlated with a wide, two-sided gap, confirming they are genuinely different quantities (numbers above — keep these for the `utilization_pct` vs. `cap_ratio` decision in §5); `design_life_years` is a constant (25) across every row, carrying no information; and `fault_cause` perfectly determines `fault_this_year == 0`, making it a leakage proxy that must be excluded from any model.

No rows were dropped in this notebook. `fault_cause` and `design_life_years` are only actually removed later, in `src/preprocessing.py` (blueprint §6, Step 3).

**Section 4 (variable meaning) is not started in this notebook.**